In [1]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from gwgraphs import *

In [2]:
def read_graph_data():
  # Read the adjacency matrix (edges)
  with open("IMDB-BINARY_A.txt", "r") as f:
      edges = [tuple(map(int, line.split(','))) for line in f]

  # Read graph indicators
  with open("IMDB-BINARY_graph_indicator.txt", "r") as f:
      graph_indicators = [int(line.strip()) for line in f if line.strip()]

  # Read graph labels
  with open("IMDB-BINARY_graph_labels.txt", "r") as f:
      graph_labels = [int(line.strip()) for line in f]

  return edges, graph_indicators, graph_labels


def construct_networkx_graph(edges, graph_indicators, graph_labels):
  graphs = {}
  for node_id, graph_id in enumerate(graph_indicators, start=1):
      if graph_id not in graphs:
          graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

  for edge in edges:
      node1, node2 = edge
      graph_id = graph_indicators[
          node1 - 1]  # Determine which graph the edge belongs to
      graphs[graph_id].add_edge(node1, node2)

  return graphs


def compute_averages(graphs):
  # Initialize variables
  total_nodes = 0
  class_nodes = {}
  class_counts = {}
  # Iterate through graphs to compute totals
  for graph_id, graph in graphs.items():
      num_nodes = graph.number_of_nodes()
      total_nodes += num_nodes

      # Get the class label for the current graph
      graph_label = graph.graph['label']

      if graph_label not in class_nodes:
          class_nodes[graph_label] = 0
          class_counts[graph_label] = 0

      class_nodes[graph_label] += num_nodes
      class_counts[graph_label] += 1
  # Calculate total average
  total_avg = round(total_nodes / len(graphs))

  # Calculate class-based averages
  class_averages = {label: round(class_nodes[label] / count)
                    for label, count in class_counts.items()}

  return total_avg, class_averages


def compute_distance_matrix(graph):
  num_nodes = len(graph.nodes)
  distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

  # Get the correct node indices to be used for indexing the matrix
  node_list = list(graph.nodes())
  node_index = {node: i for i, node in enumerate(node_list)}

  for node in graph.nodes:
      lengths = nx.shortest_path_length(graph, source=node, weight="weight")

      for target, length in lengths.items():
          if target in node_index:
              distance_matrix[node_index[node], node_index[target]] = length

  return distance_matrix

In [3]:
edges, graph_indicators, graph_labels = read_graph_data()
graphs = construct_networkx_graph(edges, graph_indicators, graph_labels)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )
total_avg, class_averages = compute_averages(graphs)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

Graph ID: 1, number of nodes: 20, label: 0
Graph ID: 2, number of nodes: 32, label: 0
Graph ID: 3, number of nodes: 21, label: 0
Graph ID: 4, number of nodes: 35, label: 0
Graph ID: 5, number of nodes: 14, label: 0
Graph ID: 6, number of nodes: 63, label: 0
Graph ID: 7, number of nodes: 12, label: 0
Graph ID: 8, number of nodes: 15, label: 0
Graph ID: 9, number of nodes: 18, label: 0
Graph ID: 10, number of nodes: 12, label: 0
Graph ID: 11, number of nodes: 12, label: 0
Graph ID: 12, number of nodes: 20, label: 0
Graph ID: 13, number of nodes: 13, label: 0
Graph ID: 14, number of nodes: 20, label: 0
Graph ID: 15, number of nodes: 72, label: 0
Graph ID: 16, number of nodes: 20, label: 0
Graph ID: 17, number of nodes: 18, label: 0
Graph ID: 18, number of nodes: 12, label: 0
Graph ID: 19, number of nodes: 19, label: 0
Graph ID: 20, number of nodes: 46, label: 0
Graph ID: 21, number of nodes: 30, label: 0
Graph ID: 22, number of nodes: 24, label: 0
Graph ID: 23, number of nodes: 22, label:

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    # Convert label 0 to -1
    if label == 0:
        label = -1
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_distance_matrix(graph) for graph in graphs_list]

# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class -1: ", labels_list.count(-1))

Elements of Class 1:  500
Elements of Class -1:  500


In [5]:
num_folds = 10
k_list = [1,2,3,4,5,6,7,8,9,10]

In [6]:
accuracies, by_class_accuracies, f1_scores, best_k_info = gw_knn(distance_matrices, labels_list, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 61.53846153846154%, Fold 0
k=1, Class -1 Accuracy: 56.25%, Fold 0
k=1, Overall Classification Accuracy: 59.0%, Fold 0
k=1, F1 Score (macro avg): 58.90%, Fold 0
k=2, Class 1 Accuracy: 61.53846153846154%, Fold 0
k=2, Class -1 Accuracy: 56.25%, Fold 0
k=2, Overall Classification Accuracy: 59.0%, Fold 0
k=2, F1 Score (macro avg): 58.90%, Fold 0
k=3, Class 1 Accuracy: 65.38461538461539%, Fold 0
k=3, Class -1 Accuracy: 54.166666666666664%, Fold 0
k=3, Overall Classification Accuracy: 60.0%, Fold 0
k=3, F1 Score (macro avg): 59.74%, Fold 0
k=4, Class 1 Accuracy: 69.23076923076923%, Fold 0
k=4, Class -1 Accuracy: 54.166666666666664%, Fold 0
k=4, Overall Classification Accuracy: 62.0%, Fold 0
k=4, F1 Score (macro avg): 61.62%, Fold 0
k=5, Class 1 Accuracy: 63.46153846153846%, Fold 0
k=5, Class -1 Accuracy: 58.333333333333336%, Fold 0
k=5, Overall Classification Accuracy: 61.0%, Fold 0
k=5, F1 Score (macro avg): 60.90%, Fold 0
k=6, Class 1 Accura

In [7]:
# ATTEMPT 2: Try adjacency matrices
def compute_adjacency_matrix(graph):
    """
    Computes the adjacency matrix for a given graph.
    
    Args:
        graph: A NetworkX graph.
    
    Returns:
        A numpy array representing the adjacency matrix where the value 
        at (i, j) denotes the presence of an edge between node i and node j.
    """
    # Initialize a matrix of zeros
    num_nodes = len(graph.nodes)
    adjacency_matrix = np.zeros((num_nodes, num_nodes))
    
    # Populate the adjacency matrix
    for i, node_i in enumerate(graph.nodes()):
        for j, node_j in enumerate(graph.nodes()):
            if graph.has_edge(node_i, node_j):
                adjacency_matrix[i, j] = 1.0
    return adjacency_matrix

In [8]:
adj_matrices = [compute_adjacency_matrix(graph) for graph in graphs_list]
accuraciesadj, by_class_accuraciesadj, f1_scoresadj, best_k_infoadj = gw_knn(adj_matrices, labels_list, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 65.38461538461539%, Fold 0
k=1, Class -1 Accuracy: 60.416666666666664%, Fold 0
k=1, Overall Classification Accuracy: 63.0%, Fold 0
k=1, F1 Score (macro avg): 62.91%, Fold 0
k=2, Class 1 Accuracy: 65.38461538461539%, Fold 0
k=2, Class -1 Accuracy: 60.416666666666664%, Fold 0
k=2, Overall Classification Accuracy: 63.0%, Fold 0
k=2, F1 Score (macro avg): 62.91%, Fold 0
k=3, Class 1 Accuracy: 57.692307692307686%, Fold 0
k=3, Class -1 Accuracy: 50.0%, Fold 0
k=3, Overall Classification Accuracy: 54.0%, Fold 0
k=3, F1 Score (macro avg): 53.83%, Fold 0
k=4, Class 1 Accuracy: 65.38461538461539%, Fold 0
k=4, Class -1 Accuracy: 54.166666666666664%, Fold 0
k=4, Overall Classification Accuracy: 60.0%, Fold 0
k=4, F1 Score (macro avg): 59.74%, Fold 0
k=5, Class 1 Accuracy: 67.3076923076923%, Fold 0
k=5, Class -1 Accuracy: 58.333333333333336%, Fold 0
k=5, Overall Classification Accuracy: 63.0%, Fold 0
k=5, F1 Score (macro avg): 62.82%, Fold 0
k=6, Cl

In [9]:
# Normalize the matrices
Ms = []
for matrix in distance_matrices:
    Ms.append(matrix / np.max(matrix))

k_bary = 20  # use overall dataset average
heights = [
    np.full(len(graph), 1 / len(graph)) for graph in graphs_list
]
num_folds = 10

accuracies2, by_class_accuracies2, f1_scores2, best_k_info2 = k_folds_glgw_knn(distance_matrices,
            labels_list,
            k_bary,
            Ms,
            heights,
            num_folds,
            seed=42,
            k_knn_list=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            method="kfold")

k-fold CV selected
Fold 0:
Time taken for reference barycenter computation: 0.7619 seconds
Time taken for LGW computation: 7.3571 seconds
k=1, Class 1 Accuracy: 50.0%
k=1, Class -1 Accuracy: 62.5%
k=1, Overall Classification Accuracy: 56.00000000000001%
k=1, F1 Score (macro avg): 55.93%
k=2, Class 1 Accuracy: 50.0%
k=2, Class -1 Accuracy: 62.5%
k=2, Overall Classification Accuracy: 56.00000000000001%
k=2, F1 Score (macro avg): 55.93%
k=3, Class 1 Accuracy: 46.15384615384615%
k=3, Class -1 Accuracy: 66.66666666666666%
k=3, Overall Classification Accuracy: 56.00000000000001%
k=3, F1 Score (macro avg): 55.72%
k=4, Class 1 Accuracy: 50.0%
k=4, Class -1 Accuracy: 58.333333333333336%
k=4, Overall Classification Accuracy: 54.0%
k=4, F1 Score (macro avg): 53.98%
k=5, Class 1 Accuracy: 46.15384615384615%
k=5, Class -1 Accuracy: 62.5%
k=5, Overall Classification Accuracy: 54.0%
k=5, F1 Score (macro avg): 53.83%
k=6, Class 1 Accuracy: 46.15384615384615%
k=6, Class -1 Accuracy: 64.58333333333334%
